# Day 31: Objects, State and Behaviour

**Today's question:** what is the dot in `name.upper()` really doing, and how could we build our *own* kinds of
values, like a bank account or a pet?

Before running a cell marked 🤔, predict what it will show.

## 1. Everything is an object

You have used objects for 30 days. Every value in Python is an **object**: it has a **type**, some **state** (the
data inside it) and **behaviour** (methods, called with a dot).

In [ ]:
name = "asha khan"
print(type(name))
print(name.title())
print(name)

`title()` gave back a **new** string. `name` itself did not change: strings can never change.

In [ ]:
marks = [72, 85]
print(type(marks))
marks.append(90)
print(marks)

`append` changed the list **itself**. That is the list's behaviour changing the list's state.

🤔 **Predict:** what does `result` hold?

In [ ]:
result = marks.append(64)
print(result)
print(marks)

`append` changes the list and returns `None`. That is why `marks = marks.append(64)` loses your list.

| Kind of method | Examples | Effect |
|----------------|----------|--------|
| changes the object | `list.append`, `list.sort`, `list.remove` | object is different afterwards, returns `None` |
| returns a new value | `str.upper`, `str.replace`, `list.count` | object unchanged, use the returned value |

🤔 **Predict** the type of each value before running.

In [ ]:
print(type(42))
print(type(3.5))
print(type(True))
print(type(None))
print(type({"owner": "Asha"}))

Even numbers have methods, and even `print` is an object:

In [ ]:
print((4.0).is_integer())
print((4.5).is_integer())
print(type(print))

Notice the word **class** in `<class 'str'>`. A class is a *kind* of object. `"asha khan"` is one object of the
class `str`. Tomorrow we write our own classes.

## 2. State and behaviour of real-world things

| Thing | State (what it *has* / *is*) | Behaviour (what it *does* / what we *do to it*) |
|-------|------------------------------|--------------------------------------------------|
| Bank account | owner (`str`), balance (`float`) | deposit, withdraw, describe |
| Pet | name (`str`), species (`str`), hunger (`int`) | feed, play, describe |
| Library book | title, author (`str`), on loan (`bool`) | borrow, give back |

Rule of thumb: **nouns and adjectives → state · verbs → behaviour.**

## 3. A bank account as a dict + functions

We already know a dict can hold the **state** and functions can hold the **behaviour**. Let's combine them.

In [ ]:
def create_account(owner, balance=0):
    """Return a new account dict."""
    return {"owner": owner, "balance": balance}


def deposit(account, amount):
    """Add money to the account."""
    if amount <= 0:
        raise ValueError("Deposit must be more than 0.")
    account["balance"] += amount


def withdraw(account, amount):
    """Take money out, but never more than the balance."""
    if amount > account["balance"]:
        raise ValueError("Not enough money in the account.")
    account["balance"] -= amount


def describe_account(account):
    return f"{account['owner']}: balance {account['balance']:.2f}"

🤔 **Predict:** `deposit` has no `return`. Will `asha` change?

In [ ]:
asha = create_account("Asha", 100)
deposit(asha, 50)
withdraw(asha, 30)
print(describe_account(asha))

Yes: a dict is mutable, and the function receives **the same dict** (Day 07 name tags), not a copy.

A second account is independent of the first:

In [ ]:
omar = create_account("Omar")
deposit(omar, 20)
print(describe_account(asha))
print(describe_account(omar))

The validation works too:

In [ ]:
withdraw(omar, 500)

It works. So what is the problem?

## 4. Where it gets messy

**Pain point 1: data and behaviour live apart.** Every single call needs the dict passed in, and nothing in the code
says that `deposit`, `withdraw` and `describe_account` belong together.

**Pain point 2: nothing protects the state.** `withdraw` refuses to go below zero, but nothing forces anyone to use
`withdraw`.

In [ ]:
asha["balance"] = -500
print(describe_account(asha))

🤔 **Predict:** we "fix" it, but with a typo in the key. What is Asha's balance now?

In [ ]:
asha["balanse"] = 1000
print(describe_account(asha))
print(asha)

No error at all. The typo silently created a **new key** that nothing ever reads.

**Pain point 3: name clashes as the program grows.** Suppose the account part of a bigger program uses the short,
natural name `describe`:

In [ ]:
def describe(account):
    return f"{account['owner']}: balance {account['balance']:.2f}"


print(describe(omar))

Later, someone adds pets to the same program and also picks the natural name `describe`:

In [ ]:
def create_pet(name, species):
    return {"name": name, "species": species, "hunger": 5}


def describe(pet):
    return f"{pet['name']} the {pet['species']}, hunger {pet['hunger']}"


rex = create_pet("Rex", "dog")
print(describe(rex))

🤔 **Predict:** what happens now with the account?

In [ ]:
print(describe(omar))

The second `def describe` **replaced** the first one. The usual workaround is to glue the type name onto every
function by hand: `describe_account`, `describe_pet`, `feed_pet`, `play_with_pet`, …

**Pain point 4: any dict fits any function.** Python cannot tell an account dict from a pet dict:

In [ ]:
deposit(rex, 10)

The mistake is only discovered when that line runs, with an error that talks about a missing key instead of
"this is not an account".

| # | Pain point | Symptom |
|---|------------|---------|
| 1 | data and behaviour live apart | the dict is passed into every function |
| 2 | nothing protects the state | negative balances, typo keys, **no error** |
| 3 | names clash as programs grow | `describe_account`, `describe_pet`, … |
| 4 | any dict fits any function | `KeyError` far from the real mistake |

## 5. Teaser: what tomorrow looks like

What if the data and the functions that work on it lived in **one bundle**, and Python built every account with the
same shape? That bundle is a **class**. Don't worry about the syntax yet; just notice `asha.owner` instead of
`asha["owner"]`.

In [ ]:
class Account:
    def __init__(self, owner):
        self.owner = owner

In [ ]:
asha = Account("Asha")
print(asha.owner)
print(type(asha))

`<class '__main__.Account'>`: we have made our **own type**, just like `str` or `list`.

## Recap

1. Every value is an object: a **type**, **state** (data) and **behaviour** (methods).
2. Some methods change the object (lists); others return a new value (strings).
3. A dict + functions can model a thing, but the state is unprotected, every function needs the dict, names clash,
   and any dict fits any function.

**Next time:** `class`, `__init__`, `self` and attributes: writing our own types.